In [ ]:
from crewai import LLM
llm = LLM(model="ollama/llama3.2", base_url="http://localhost:11434")

In [ ]:
#define the agents and tasks in the yaml file

import yaml
with open('config/planner_agents.yaml', 'r') as f: # open the file-> 'r' means read mode --> 'f' closes the file even if there are errors
    agent_config = yaml.safe_load(f)        # you need config file to run the code, it act as a variable 
                                    # while yaml.safe.load() is use to read the yaml file and it converts into python directory
with open('config/planner_tasks.yaml','r') as f:
    tasks_config = yaml.safe_load(f)


In [ ]:
from pydantic import BaseModel #pydantic is the library used for data validation whereas basemodel is the base class for creating data models like tweet and threads
from typing import Optional   #typing provides the vocabulary (str, Optional, list[str]) 

class Tweet(BaseModel):
    """Represents an individual tweet in a thread"""
    content: str
    is_hook: bool = False   #indicates whether the tweet is a hook (attention-grabbing) or not
    media_urls: Optional[list[str]] = []    #here multiple media urls are allowed for single tweet.

class Thread(BaseModel):
    """Represents a Twitter thread"""
    topic: str
    tweets: list[Tweet]

In [ ]:
from pydantic import BaseModel
from typing import Optional

class LinkedInPost(BaseModel):
    """Represents a Linkedin post"""
    content: str
    media_url: str # only one url allowed for linkedin post

In [ ]:
#each agent has to read some files for processing the data. eg: draft analyzer agent must analyze the scraped draft.

from crewai_tools import DirectoryReadTool, FileReadTool
all_tools = [DirectoryReadTool(), FileReadTool()]


In [ ]:
from crewai import Task, Agent 

draft_analyzer = Agent(config=agent_config['draft_analyzer'], tools=all_tools, llm=llm)
analyze_draft = Task(config=tasks_config['analyze_draft'], agent=draft_analyzer)

In [ ]:
from crewai import Agent, Task

twitter_thread_planner = Agent(config=agent_config['twitter_thread_planner'],
                               tools=all_tools,
                               llm=llm)

create_twitter_thread_plan = Task(config=tasks_config['create_twitter_thread_plan'],
                                  agent=twitter_thread_planner,
                                  output_pydantic=Thread)


In [ ]:
from crewai import Agent, Task

linkedin_post_planner = Agent(config=agent_config['linkedin_post_planner'],
                              tools=all_tools,
                              llm=llm)

create_linkedin_post_plan = Task(config=tasks_config['create_linkedin_post_plan'],
                                 agent=linkedin_post_planner,
                                 output_pydantic=LinkedInPost)
